# Manifest Reconstruction - Real Dataset Demo

This notebook demonstrates the **manifest reconstruction** feature that rebuilds `verification_manifest.json` from cache database and filesystem when the original manifest is lost or corrupted.

**Required Files**:
- `notebooks/data/products.parquet` - Dataset file (for validation)
- `notebooks/data/verification_manifest.json` - Original manifest (will back up and delete)
- `output/download_cache.db` - Cache database (contains product metadata)
- `data/images/` - Filesystem structure (category/series/product hierarchy)

## Features Demonstrated
1. Simulating manifest loss (backup and delete)
2. Reconstructing manifest from cache + filesystem
3. Comparing reconstructed vs original manifest
4. Validating dataset with reconstructed manifest
5. CLI command usage
6. Edge cases and error handling

In [ ]:
# Imports
import json
import shutil
from pathlib import Path

from loguru import logger

import os
import sys

# Add the project root to the python path
# ruff: noqa: E402
root_path = Path(os.getcwd()).parent
# ruff: noqa: E402
if str(root_path) not in sys.path:
    sys.path.append(str(root_path))

from src.building.manifest_reconstruction import ManifestReconstructor
from src.validation.validator import DatasetValidator

# Configure logger for notebook
logger.remove()
logger.add(
    lambda msg: print(msg, end=""), colorize=True, format="<level>{message}</level>"
)

## Step 1: Setup Paths and Verify Prerequisites

In [ ]:
# Paths
manifest_path = Path("notebooks/data/verification_manifest.json")
manifest_backup_path = Path("notebooks/data/verification_manifest_original_backup.json")
reconstructed_manifest_path = Path(
    "notebooks/data/verification_manifest_reconstructed.json"
)
dataset_path = Path("notebooks/data/products.parquet")

# Cache and data directories (production locations)
cache_db_path = Path("output/download_cache.db")
data_dir = Path("data/")

print("📁 File Status:")
print(
    f"  Manifest: {manifest_path} - {'✓ Exists' if manifest_path.exists() else '✗ Missing'}"
)
print(
    f"  Dataset: {dataset_path} - {'✓ Exists' if dataset_path.exists() else '✗ Missing'}"
)
print(
    f"  Cache DB: {cache_db_path} - {'✓ Exists' if cache_db_path.exists() else '✗ Missing'}"
)
print(f"  Data dir: {data_dir} - {'✓ Exists' if data_dir.exists() else '✗ Missing'}")

if not manifest_path.exists():
    print(
        "\n⚠️  Warning: Original manifest not found. Follow README setup instructions."
    )
if not cache_db_path.exists():
    print("\n⚠️  Warning: Cache database not found. Run 'cidar-scrape' first.")
if not data_dir.exists():
    print("\n⚠️  Warning: Data directory not found. Run 'cidar-scrape' first.")

## Step 2: Inspect Original Manifest

Let's examine the structure and stats of the original manifest before reconstruction.

In [ ]:
if manifest_path.exists():
    with open(manifest_path, "r", encoding="utf-8") as f:
        original_manifest = json.load(f)

    print("📊 Original Manifest Statistics:")
    print(f"  Total Categories: {original_manifest.get('total_categories', 0)}")
    print(f"  Total Series: {original_manifest.get('total_series', 0)}")
    print(f"  Total Products: {original_manifest.get('total_products', 0)}")
    print(f"  Total Images: {original_manifest.get('total_images', 0)}")
    print(f"  Total PDFs: {original_manifest.get('total_pdfs_with_urls', 0)}")
    print(f"  Total Specs: {original_manifest.get('total_specs_populated', 0)}")
    print(f"  Scrape Timestamp: {original_manifest.get('scrape_timestamp', 'N/A')}")
    print(f"  Reconstructed: {original_manifest.get('reconstructed', False)}")

    # Sample a few categories
    print("\n📁 Sample Categories:")
    categories = original_manifest.get("categories", {})
    for cat_name in list(categories.keys())[:3]:
        series_count = len(categories[cat_name].get("series", {}))
        print(f"  - {cat_name}: {series_count} series")
else:
    print("❌ Original manifest not found. Cannot proceed with demo.")

## Step 3: Create Backup and Simulate Manifest Loss

Back up the original manifest, then delete it to simulate loss/corruption.

In [ ]:
if manifest_path.exists():
    # Create backup
    shutil.copy2(manifest_path, manifest_backup_path)
    print(f"✓ Backup created: {manifest_backup_path}")
    print(f"  Size: {manifest_backup_path.stat().st_size / 1024:.1f} KB")

    # Delete original to simulate loss
    manifest_path.unlink()
    print(f"\n🗑️  Deleted original manifest: {manifest_path}")
    print(
        f"  Status: {'✓ Deleted' if not manifest_path.exists() else '✗ Still exists'}"
    )

    print("\n⚠️  Manifest is now MISSING - dataset cannot be validated without it!")
else:
    print("❌ No manifest to backup")

## Step 4: Reconstruct Manifest from Cache + Filesystem

Use `ManifestReconstructor` to rebuild the manifest from:
1. Filesystem structure (`data/images/` hierarchy)
2. Product cache database (`output/download_cache.db`)

In [ ]:
# Initialize reconstructor with custom output path
reconstructor = ManifestReconstructor(
    cache_db_path=cache_db_path,
    data_dir=data_dir,
    output_path=reconstructed_manifest_path,  # Save to separate file for comparison
)

print("🔧 Reconstructing manifest...\n")
success = reconstructor.reconstruct()

print(
    f"\n{'✓' if success else '✗'} Reconstruction result: {'SUCCESS' if success else 'FAILED'}"
)

if success:
    print(f"\n📄 Reconstructed manifest saved to: {reconstructed_manifest_path}")
    print(f"  File size: {reconstructed_manifest_path.stat().st_size / 1024:.1f} KB")

## Step 5: Compare Original vs Reconstructed Manifest

Verify reconstruction accuracy by comparing statistics and structure.

In [ ]:
if reconstructed_manifest_path.exists() and manifest_backup_path.exists():
    # Load both manifests
    with open(manifest_backup_path, "r", encoding="utf-8") as f:
        original = json.load(f)

    with open(reconstructed_manifest_path, "r", encoding="utf-8") as f:
        reconstructed = json.load(f)

    print("📊 Comparison: Original vs Reconstructed\n")

    metrics = [
        "total_categories",
        "total_series",
        "total_products",
        "total_images",
        "total_pdfs_with_urls",
        "total_specs_populated",
    ]

    print(f"{'Metric':<25} {'Original':<12} {'Reconstructed':<15} {'Match':<10}")
    print("-" * 65)

    for metric in metrics:
        orig_val = original.get(metric, 0)
        recon_val = reconstructed.get(metric, 0)
        match = "✓" if orig_val == recon_val else "✗"
        print(f"{metric:<25} {orig_val:<12} {recon_val:<15} {match:<10}")

    # Check reconstructed flag
    print("\n🔍 Metadata:")
    print(f"  Original 'reconstructed' flag: {original.get('reconstructed', False)}")
    print(
        f"  Reconstructed 'reconstructed' flag: {reconstructed.get('reconstructed', False)}"
    )
    print(
        f"  Reconstruction timestamp: {reconstructed.get('reconstruction_timestamp', 'N/A')}"
    )

    # Sample category comparison
    print("\n📁 Category Comparison:")
    orig_categories = set(original.get("categories", {}).keys())
    recon_categories = set(reconstructed.get("categories", {}).keys())

    print(f"  Original categories: {len(orig_categories)}")
    print(f"  Reconstructed categories: {len(recon_categories)}")
    print(f"  Match: {'✓' if orig_categories == recon_categories else '✗'}")

    if orig_categories != recon_categories:
        missing = orig_categories - recon_categories
        extra = recon_categories - orig_categories
        if missing:
            print(f"\n  ⚠️  Missing categories: {missing}")
        if extra:
            print(f"\n  ⚠️  Extra categories: {extra}")
else:
    print("❌ Cannot compare - missing files")

## Step 6: Validate Dataset with Reconstructed Manifest

Test that the reconstructed manifest can be used for dataset validation.

In [ ]:
if reconstructed_manifest_path.exists() and dataset_path.exists():
    print("🔍 Validating dataset with reconstructed manifest...\n")

    validator = DatasetValidator(
        parquet_path=dataset_path,
        manifest_path=reconstructed_manifest_path,
    )

    success = validator.validate_all(verbose=False, project_root=Path.cwd())

    print(
        f"\n{'✓' if success else '✗'} Validation result: {'PASS' if success else 'FAIL'}"
    )
    print(f"  Errors: {len(validator.errors)}")
    print(f"  Warnings: {len(validator.warnings)}")

    if success:
        print("\n✅ Reconstructed manifest is valid and can be used for validation!")
    else:
        print("\n⚠️  Validation found issues - check error details above.")
else:
    print("❌ Cannot validate - missing files")

## Step 7: Restore Original Manifest

Copy the reconstructed manifest back to the original location.

In [ ]:
if reconstructed_manifest_path.exists():
    # Copy reconstructed manifest to original location
    shutil.copy2(reconstructed_manifest_path, manifest_path)
    print(f"✓ Restored manifest: {manifest_path}")
    print(f"  Source: {reconstructed_manifest_path}")
    print(f"  Status: {'✓ Exists' if manifest_path.exists() else '✗ Failed'}")

    # Verify it's usable
    with open(manifest_path, "r", encoding="utf-8") as f:
        restored_manifest = json.load(f)

    print("\n📊 Restored Manifest:")
    print(f"  Products: {restored_manifest.get('total_products', 0)}")
    print(f"  Reconstructed flag: {restored_manifest.get('reconstructed', False)}")
else:
    print("❌ Reconstructed manifest not found")

## Step 8: CLI Command Demonstration

Show how to use the `cidar-reconstruct-manifest` CLI command.

In [ ]:
print("💻 CLI Command Usage:\n")
print("# Basic usage (uses default paths)")
print("cidar-reconstruct-manifest\n")

print("# Custom paths")
print("cidar-reconstruct-manifest \\")
print("  --cache-db output/download_cache.db \\")
print("  --data-dir data/ \\")
print("  --output output/verification_manifest.json \\")
print("  --force\n")

print("# For this demo (reconstruct to notebooks/data/)")
print("cidar-reconstruct-manifest \\")
print("  --cache-db output/download_cache.db \\")
print("  --data-dir data/ \\")
print("  --output notebooks/data/verification_manifest.json \\")
print("  --force")

## Step 9: Test Edge Cases

Demonstrate behavior when cache or filesystem is incomplete.

In [ ]:
print("🔬 Edge Case Testing:\n")

# Test 1: Missing cache database
print("Test 1: What happens if cache database is missing?")
fake_reconstructor = ManifestReconstructor(
    cache_db_path=Path("output/nonexistent_cache.db"),
    data_dir=data_dir,
    output_path=Path("notebooks/data/test_manifest.json"),
)
print("  Expected: Reconstruction continues with filesystem data only")
print("  Product metadata (URLs, specs) will be missing\n")

# Test 2: Empty data directory
print("Test 2: What happens if data directory is empty?")
fake_reconstructor2 = ManifestReconstructor(
    cache_db_path=cache_db_path,
    data_dir=Path("nonexistent_data/"),
    output_path=Path("notebooks/data/test_manifest2.json"),
)
print("  Expected: Reconstruction fails (no products found)")
print("  Error logged: 'No data found in filesystem'\n")

print("✓ Edge cases documented (not executed to avoid errors)")

## Step 10: Cleanup Test Files

Remove temporary files created during the demo.

In [ ]:
print("🗑️  Cleaning up test files...\n")

files_to_remove = [
    reconstructed_manifest_path,
    manifest_backup_path,
]

removed_count = 0
for file_path in files_to_remove:
    if file_path.exists():
        file_path.unlink()
        removed_count += 1
        print(f"  ✓ Removed: {file_path.name}")

print(f"\n✓ Cleanup complete: {removed_count} files removed")
print("\n📌 Note: Original manifest has been restored from reconstruction")

## Summary

This notebook demonstrated:

1. ✅ **Simulating manifest loss** - Backed up and deleted original manifest
2. ✅ **Manifest reconstruction** - Rebuilt from cache database + filesystem
3. ✅ **Accuracy verification** - Compared statistics and structure
4. ✅ **Validation integration** - Used reconstructed manifest for dataset validation
5. ✅ **CLI command usage** - Demonstrated `cidar-reconstruct-manifest` syntax
6. ✅ **Edge case handling** - Documented behavior with missing data

### Key Features

**Reconstruction Strategy:**
1. Scans `data/images/` filesystem to discover hierarchy:
   ```
   data/images/{VENDOR}/{CATEGORY}/{SERIES}/{CAMERA_ID}/
   ```
2. Loads product metadata from `output/download_cache.db` (product_cache table)
3. Combines both sources to rebuild complete manifest
4. Marks manifest with `"reconstructed": true` flag

**When to Use Manifest Reconstruction:**
- Manifest file deleted or corrupted
- Scraping completed but manifest creation failed
- Need to regenerate manifest from existing data
- Recovery after filesystem/database issues

**Limitations:**
- Requires cache database (`download_cache.db`) for full metadata
- Without cache: filesystem structure is preserved but URLs/specs may be missing
- Cannot recover data that was never scraped

### CLI Quick Reference

```bash
# Basic reconstruction (uses default paths)
cidar-reconstruct-manifest

# Custom paths
cidar-reconstruct-manifest \
  --cache-db output/download_cache.db \
  --data-dir data/ \
  --output output/verification_manifest.json \
  --force

# Then validate the reconstructed manifest
cidar-validate
```

### Integration with Other Phases

**Recovery Workflow:**
```bash
# 1. Scraping completed but manifest lost
cidar-scrape  # Already done

# 2. Reconstruct manifest from cache + filesystem
cidar-reconstruct-manifest

# 3. Build dataset with reconstructed manifest
cidar-build

# 4. Validate dataset
cidar-validate
```